In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Silver layer - Clean marathon OBT
# MAGIC
# MAGIC This notebook creates the cleaned Silver table:
# MAGIC
# MAGIC `marathos.silver.races_obt`
# MAGIC
# MAGIC Bronze keeps the raw CSV data mostly unchanged.
# MAGIC Silver cleans the data so it can be used for Gold tables, dashboard and Genie.
# MAGIC
# MAGIC Cleaning decisions:
# MAGIC - Keep events from 2010 onward.
# MAGIC - Remove rows without event name, country, gender, distance or performance.
# MAGIC - Remove multi-day/stage events because they are harder to compare fairly.
# MAGIC - Keep only km and mi distance values.
# MAGIC - Convert miles to kilometres.
# MAGIC - Convert performance time to seconds.
# MAGIC - Calculate speed from distance and performance time.
# MAGIC - Remove unrealistic speeds over 25 km/h.
# MAGIC - Use sha2 hash IDs instead of dense_rank.

# COMMAND ----------

from pyspark.sql import functions as F

# Source and target tables
BRONZE_TABLE = "marathos.bronze.races_raw"
SILVER_TABLE = "marathos.silver.races_obt"

# COMMAND ----------

# Read Bronze data
bronze = spark.table(BRONZE_TABLE)

display(bronze.limit(10))

print("Bronze rows:", bronze.count())
print("Bronze columns:", len(bronze.columns))
bronze.printSchema()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Rename raw columns
# MAGIC
# MAGIC The original CSV has beginner-unfriendly column names with spaces.
# MAGIC I rename them to snake_case so they are easier to use in Python and SQL.

# COMMAND ----------

df = (
    bronze
    .select(
        F.col("year_of_event").alias("event_year_raw"),
        F.col("event_dates").alias("event_dates_raw"),
        F.col("event_name").alias("event_name_raw"),
        F.col("event_distance_length").alias("event_distance_raw"),
        F.col("event_number_of_finishers").alias("event_finishers_raw"),
        F.col("athlete_performance").alias("performance_raw"),
        F.col("athlete_club").alias("athlete_club_raw"),
        F.col("athlete_country").alias("athlete_country_raw"),
        F.col("athlete_year_of_birth").alias("athlete_birth_year_raw"),
        F.col("athlete_gender").alias("athlete_gender_raw"),
        F.col("athlete_age_category").alias("athlete_age_category_raw"),
        F.col("athlete_average_speed").alias("athlete_average_speed_raw"),
        F.col("athlete_id_raw").alias("athlete_source_id")
    )
)

display(df.limit(10))
df.printSchema()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Basic text cleaning
# MAGIC
# MAGIC I trim spaces and convert empty strings to null.

# COMMAND ----------

def clean_text(column_name):
    return F.when(
        F.trim(F.col(column_name).cast("string")) == "",
        F.lit(None)
    ).otherwise(F.trim(F.col(column_name).cast("string")))

df = (
    df
    .withColumn("event_name", clean_text("event_name_raw"))
    .withColumn("event_dates", clean_text("event_dates_raw"))
    .withColumn("event_distance_raw", clean_text("event_distance_raw"))
    .withColumn("performance_raw", clean_text("performance_raw"))
    .withColumn("athlete_club", clean_text("athlete_club_raw"))
    .withColumn("athlete_country", F.upper(clean_text("athlete_country_raw")))
    .withColumn("athlete_gender", F.upper(clean_text("athlete_gender_raw")))
    .withColumn("athlete_age_category", clean_text("athlete_age_category_raw"))
)

display(df.limit(10))

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Convert year and finishers safely
# MAGIC
# MAGIC Some values can contain strange text. I use `regexp_extract` instead of direct cast,
# MAGIC so malformed values do not crash the notebook.

# COMMAND ----------

df = (
    df
    .withColumn(
        "event_year",
        F.regexp_extract(F.col("event_year_raw").cast("string"), r"(\d{4})", 1).cast("int")
    )
    .withColumn(
        "event_finishers",
        F.regexp_replace(
            F.regexp_extract(F.col("event_finishers_raw").cast("string"), r"(\d+)", 1),
            ",",
            ""
        ).cast("int")
    )
    .withColumn(
        "athlete_birth_year",
        F.regexp_extract(F.col("athlete_birth_year_raw").cast("string"), r"(\d{4})", 1).cast("int")
    )
)

display(
    df.select(
        "event_year_raw",
        "event_year",
        "event_finishers_raw",
        "event_finishers",
        "athlete_birth_year_raw",
        "athlete_birth_year"
    ).limit(20)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Clean distance
# MAGIC
# MAGIC The distance column can contain values like `50km`, `100km`, `26mi`.
# MAGIC I extract the number and unit separately.
# MAGIC Miles are converted to kilometres.

# COMMAND ----------

df = (
    df
    .withColumn("distance_text", F.lower(F.col("event_distance_raw")))
    .withColumn(
        "distance_value",
        F.regexp_extract(F.col("distance_text"), r"([0-9]+(?:\.[0-9]+)?)", 1).cast("double")
    )
    .withColumn(
        "distance_unit",
        F.when(F.col("distance_text").rlike("km"), F.lit("km"))
         .when(F.col("distance_text").rlike("mi|mile"), F.lit("mi"))
         .otherwise(F.lit(None))
    )
    .withColumn(
        "distance_km",
        F.when(F.col("distance_unit") == "km", F.col("distance_value"))
         .when(F.col("distance_unit") == "mi", F.col("distance_value") * F.lit(1.60934))
         .otherwise(F.lit(None))
    )
)

display(
    df.select(
        "event_distance_raw",
        "distance_value",
        "distance_unit",
        "distance_km"
    ).limit(30)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Convert performance time to seconds
# MAGIC
# MAGIC Performance can look like:
# MAGIC
# MAGIC - `2:45:30`
# MAGIC - `14:38:08 h`
# MAGIC - `45:20`
# MAGIC - bad values like `2d 00`
# MAGIC
# MAGIC I only keep normal hour/minute/second patterns. Other strange values become null.

# COMMAND ----------

time_clean = F.regexp_replace(F.lower(F.col("performance_raw")), "h", "")
time_clean = F.regexp_replace(time_clean, " ", "")

df = (
    df
    .withColumn("performance_clean", time_clean)
    .withColumn("time_parts", F.split(F.col("performance_clean"), ":"))
    .withColumn("time_part_count", F.size(F.col("time_parts")))
    .withColumn(
        "performance_seconds",
        F.when(
            F.col("performance_clean").rlike(r"^\d+:\d{1,2}:\d{1,2}$"),
            (F.col("time_parts").getItem(0).cast("int") * 3600)
            + (F.col("time_parts").getItem(1).cast("int") * 60)
            + F.col("time_parts").getItem(2).cast("int")
        )
        .when(
            F.col("performance_clean").rlike(r"^\d+:\d{1,2}$"),
            (F.col("time_parts").getItem(0).cast("int") * 60)
            + F.col("time_parts").getItem(1).cast("int")
        )
        .otherwise(F.lit(None))
    )
)

display(
    df.select(
        "performance_raw",
        "performance_clean",
        "performance_seconds"
    ).limit(30)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Calculate speed
# MAGIC
# MAGIC Speed is calculated from distance and performance:
# MAGIC
# MAGIC `speed_kmh = distance_km / hours`
# MAGIC
# MAGIC I do not trust the original average speed fully because the teacher said it contains many errors.

# COMMAND ----------

df = (
    df
    .withColumn(
        "speed_kmh",
        F.when(
            (F.col("performance_seconds") > 0) & (F.col("distance_km") > 0),
            F.col("distance_km") / (F.col("performance_seconds") / F.lit(3600.0))
        ).otherwise(F.lit(None))
    )
    .withColumn("speed_kmh", F.round(F.col("speed_kmh"), 2))
)

display(
    df.select(
        "event_name",
        "event_distance_raw",
        "distance_km",
        "performance_raw",
        "performance_seconds",
        "speed_kmh"
    ).limit(30)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Create athlete age
# MAGIC
# MAGIC Age is calculated only when birth year looks realistic.

# COMMAND ----------

df = (
    df
    .withColumn(
        "athlete_age",
        F.when(
            (F.col("athlete_birth_year").between(1900, 2015))
            & (F.col("event_year").between(2010, 2030)),
            F.col("event_year") - F.col("athlete_birth_year")
        ).otherwise(F.lit(None))
    )
)

display(
    df.select(
        "event_year",
        "athlete_birth_year",
        "athlete_age"
    ).limit(20)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Remove invalid and out-of-scope rows
# MAGIC
# MAGIC I remove:
# MAGIC
# MAGIC - rows before 2010
# MAGIC - rows missing event name, country, gender, distance or performance
# MAGIC - multi-day/stage events
# MAGIC - unrealistic distances
# MAGIC - unrealistic speeds above 25 km/h
# MAGIC - invalid ages
# MAGIC
# MAGIC This follows the teacher's advice: it is okay to remove bad or out-of-scope data if the decision is documented.

# COMMAND ----------

silver = (
    df
    .filter(F.col("event_year") >= 2010)
    .filter(F.col("event_name").isNotNull())
    .filter(F.col("athlete_country").isNotNull())
    .filter(F.col("athlete_gender").isNotNull())
    .filter(F.col("distance_km").isNotNull())
    .filter(F.col("performance_seconds").isNotNull())
    .filter(F.col("distance_km").between(40, 350))
    .filter(F.col("speed_kmh").between(2, 25))
    .filter((F.col("athlete_age").isNull()) | (F.col("athlete_age").between(10, 90)))
    .filter(~F.lower(F.col("event_name")).rlike("etappen|stage|stages|multi[- ]?day|day [0-9]|dels"))
)

display(silver.limit(20))

print("Rows after Silver cleaning:", silver.count())

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. Create stable hash IDs
# MAGIC
# MAGIC The teacher mentioned that `dense_rank()` may not work well in some pipeline/streaming-style cases.
# MAGIC Therefore I use `sha2` hash values for stable IDs.

# COMMAND ----------

silver = (
    silver
    .withColumn(
        "event_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("event_name"), F.lit("unknown_event")),
                F.coalesce(F.col("event_year").cast("string"), F.lit("unknown_year")),
                F.coalesce(F.round(F.col("distance_km"), 2).cast("string"), F.lit("unknown_distance"))
            ),
            256
        )
    )
    .withColumn(
        "athlete_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("athlete_source_id").cast("string"), F.lit("unknown_source_id")),
                F.coalesce(F.col("athlete_country"), F.lit("unknown_country")),
                F.coalesce(F.col("athlete_gender"), F.lit("unknown_gender")),
                F.coalesce(F.col("athlete_birth_year").cast("string"), F.lit("unknown_birth_year"))
            ),
            256
        )
    )
    .withColumn(
        "result_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.coalesce(F.col("event_name"), F.lit("unknown_event")),
                F.coalesce(F.col("event_year").cast("string"), F.lit("unknown_year")),
                F.coalesce(F.col("athlete_source_id").cast("string"), F.lit("unknown_source_id")),
                F.coalesce(F.col("performance_seconds").cast("string"), F.lit("unknown_performance")),
                F.coalesce(F.col("athlete_country"), F.lit("unknown_country"))
            ),
            256
        )
    )
)

display(
    silver.select(
        "result_id",
        "event_id",
        "athlete_id",
        "event_name",
        "event_year",
        "athlete_country",
        "performance_seconds"
    ).limit(10)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 10. Select final Silver columns
# MAGIC
# MAGIC This is one big cleaned table, also called an OBT: One Big Table.
# MAGIC It will be used to build the Gold fact and dimension tables.

# COMMAND ----------

silver_final = (
    silver
    .select(
        "result_id",
        "event_id",
        "athlete_id",
        "event_year",
        "event_dates",
        "event_name",
        "event_distance_raw",
        F.round(F.col("distance_km"), 3).alias("distance_km"),
        "distance_unit",
        "event_finishers",
        "performance_raw",
        "performance_seconds",
        "speed_kmh",
        "athlete_source_id",
        "athlete_club",
        "athlete_country",
        "athlete_birth_year",
        "athlete_gender",
        "athlete_age_category",
        "athlete_age"
    )
)

display(silver_final.limit(20))

print("Final Silver rows:", silver_final.count())
print("Final Silver columns:", len(silver_final.columns))
silver_final.printSchema()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 11. Save Silver table

# COMMAND ----------

(
    silver_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 12. Validate saved Silver table

# COMMAND ----------

saved_silver = spark.table(SILVER_TABLE)

display(saved_silver.limit(20))

print("Saved Silver rows:", saved_silver.count())
saved_silver.printSchema()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 13. Simple validation queries
# MAGIC
# MAGIC These checks prove that the Silver layer is usable for analysis.

# COMMAND ----------

display(
    spark.sql(f"""
        SELECT event_year, COUNT(*) AS result_count
        FROM {SILVER_TABLE}
        GROUP BY event_year
        ORDER BY event_year
    """)
)

# COMMAND ----------

display(
    spark.sql(f"""
        SELECT athlete_country, COUNT(*) AS result_count
        FROM {SILVER_TABLE}
        GROUP BY athlete_country
        ORDER BY result_count DESC
        LIMIT 15
    """)
)

# COMMAND ----------

display(
    spark.sql(f"""
        SELECT athlete_gender, COUNT(*) AS result_count
        FROM {SILVER_TABLE}
        GROUP BY athlete_gender
        ORDER BY result_count DESC
    """)
)

# COMMAND ----------

display(
    spark.sql(f"""
        SELECT distance_unit, COUNT(*) AS result_count
        FROM {SILVER_TABLE}
        GROUP BY distance_unit
        ORDER BY result_count DESC
    """)
)